# Logistic Regression Model Selection

Fit and evaluate a logistic regression model from `diabetes.csv` using the same cleaning idea from `cleaning_and_scaling.ipynb`, but without leakage. Invalid zero values are replaced with training-fold medians and scaling is learned inside the cross-validation pipeline.

In [1]:
import numpy as np
import pandas as pd

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_recall_curve,
    roc_auc_score,
)
from sklearn.model_selection import GridSearchCV, StratifiedKFold, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

RANDOM_STATE = 42

## Cleaning logic checked

In `cleaning_and_scaling.ipynb`, zeros were treated as invalid for `Glucose`, `BloodPressure`, `SkinThickness`, `Insulin`, and `BMI`, then replaced with each column median. The notebook also scaled all features before saving `diabetes_cleaned_scaled.csv`.

Here, the same zero-to-median replacement and standard scaling are moved inside the model pipeline. That means each CV fold learns medians and scaling parameters only from its own training split.

In [2]:
raw_data = pd.read_csv("diabetes.csv")
print(raw_data.shape)
raw_data.head()

(768, 9)


,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
0,6,148,72,35,0,33.6,0.627,50,1
1,1,85,66,29,0,26.6,0.351,31,0
2,8,183,64,0,0,23.3,0.672,32,1
3,1,89,66,23,94,28.1,0.167,21,0
4,0,137,40,35,168,43.1,2.288,33,1


In [3]:
zero_as_missing_columns = [
    "Glucose",
    "BloodPressure",
    "SkinThickness",
    "Insulin",
    "BMI",
]

raw_data.eq(0).sum()

Pregnancies                 111
Glucose                       5
BloodPressure                35
SkinThickness               227
Insulin                     374
BMI                          11
DiabetesPedigreeFunction      0
Age                           0
Outcome                     500
dtype: int64

In [4]:
class ZeroToMedianImputer(BaseEstimator, TransformerMixin):
    """Replace invalid zeros in selected columns with training medians."""

    def __init__(self, columns):
        self.columns = columns

    def fit(self, X, y=None):
        X = pd.DataFrame(X).copy()
        self.feature_names_in_ = X.columns.to_numpy()
        self.medians_ = {}

        for column in self.columns:
            non_zero_values = X.loc[X[column] != 0, column]
            self.medians_[column] = non_zero_values.median()

        return self

    def transform(self, X):
        X = pd.DataFrame(X, columns=self.feature_names_in_).copy()

        for column, median in self.medians_.items():
            X[column] = X[column].replace(0, median)

        return X

In [5]:
X = raw_data.drop(columns="Outcome")
y = raw_data["Outcome"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    stratify=y,
    random_state=RANDOM_STATE,
)

print("Train class balance:")
print(y_train.value_counts(normalize=True).rename("proportion"))
print("\nTest class balance:")
print(y_test.value_counts(normalize=True).rename("proportion"))

Train class balance:
0    0.651466
1    0.348534
Name: proportion, dtype: float64

Test class balance:
0    0.649351
1    0.350649
Name: proportion, dtype: float64


## Cross-validated hyperparameter search

`GridSearchCV` evaluates logistic regression settings with 5-fold stratified CV. The pipeline avoids leakage because median replacement and scaling are refit separately inside each training fold.

In [6]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

C_values = np.logspace(-3, 3, 13)

pipeline = Pipeline(
    steps=[
        ("zero_imputer", ZeroToMedianImputer(columns=zero_as_missing_columns)),
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(max_iter=10_000, random_state=RANDOM_STATE)),
    ]
)

param_grid = [
    {
        "model__solver": ["liblinear"],
        "model__penalty": ["l1", "l2"],
        "model__C": C_values,
        "model__class_weight": [None, "balanced"],
    },
    {
        "model__solver": ["lbfgs", "newton-cg", "sag"],
        "model__penalty": ["l2"],
        "model__C": C_values,
        "model__class_weight": [None, "balanced"],
    },
    {
        "model__solver": ["saga"],
        "model__penalty": ["l1", "l2"],
        "model__C": C_values,
        "model__class_weight": [None, "balanced"],
    },
    {
        "model__solver": ["saga"],
        "model__penalty": ["elasticnet"],
        "model__l1_ratio": [0.1, 0.5, 0.9],
        "model__C": C_values,
        "model__class_weight": [None, "balanced"],
    },
]

search = GridSearchCV(
    estimator=pipeline,
    param_grid=param_grid,
    scoring={"roc_auc": "roc_auc", "f1": "f1", "accuracy": "accuracy"},
    refit="roc_auc",
    cv=cv,
    n_jobs=-1,
    return_train_score=True,
)

search.fit(X_train, y_train)

print("Best CV ROC AUC:", round(search.best_score_, 4))
print("Best parameters:")
search.best_params_

Best CV ROC AUC: 0.8455
Best parameters:


{'model__C': 0.31622776601683794,
 'model__class_weight': 'balanced',
 'model__l1_ratio': 0.1,
 'model__penalty': 'elasticnet',
 'model__solver': 'saga'}

In [7]:
cv_results = pd.DataFrame(search.cv_results_)

ranked_results = (
    cv_results[
        [
            "rank_test_roc_auc",
            "mean_test_roc_auc",
            "std_test_roc_auc",
            "mean_test_f1",
            "mean_test_accuracy",
            "param_model__solver",
            "param_model__penalty",
            "param_model__C",
            "param_model__class_weight",
            "param_model__l1_ratio",
        ]
    ]
    .sort_values("rank_test_roc_auc")
    .head(10)
)

ranked_results

,rank_test_roc_auc,mean_test_roc_auc,std_test_roc_auc,mean_test_f1,mean_test_accuracy,param_model__solver,param_model__penalty,param_model__C,param_model__class_weight,param_model__l1_ratio
215,1,0.845504,0.014963,0.672385,0.762202,saga,elasticnet,0.316228,balanced,0.1
210,2,0.845443,0.015498,0.679731,0.767093,saga,elasticnet,0.100000,balanced,0.5
152,3,0.845437,0.015626,0.678417,0.767093,saga,l1,0.316228,balanced,NaN
22,3,0.845437,0.015724,0.679635,0.762215,liblinear,l1,0.316228,balanced,NaN
85,5,0.845155,0.014717,0.672385,0.762202,lbfgs,l2,0.316228,balanced,NaN
87,6,0.845097,0.014694,0.672385,0.762202,sag,l2,0.316228,balanced,NaN
153,6,0.845097,0.014694,0.672385,0.762202,saga,l2,0.316228,balanced,NaN
86,6,0.845097,0.014694,0.672385,0.762202,newton-cg,l2,0.316228,balanced,NaN
23,9,0.845097,0.014636,0.672385,0.762202,liblinear,l2,0.316228,balanced,NaN
209,10,0.845058,0.012864,0.681057,0.767093,saga,elasticnet,0.100000,balanced,0.1


## Held-out test performance

The default 0.50 threshold is reported, and a second threshold is selected on the training set to maximize F1. The test set remains held out until this final evaluation.

In [8]:
best_model = search.best_estimator_

y_train_proba = best_model.predict_proba(X_train)[:, 1]
y_test_proba = best_model.predict_proba(X_test)[:, 1]

precision, recall, thresholds = precision_recall_curve(y_train, y_train_proba)
f1_scores = 2 * precision * recall / (precision + recall + 1e-12)
best_threshold = thresholds[np.argmax(f1_scores[:-1])]

metrics = pd.DataFrame(
    {
        "threshold": [0.50, best_threshold],
        "roc_auc": [roc_auc_score(y_test, y_test_proba)] * 2,
        "accuracy": [
            accuracy_score(y_test, y_test_proba >= 0.50),
            accuracy_score(y_test, y_test_proba >= best_threshold),
        ],
        "f1": [
            f1_score(y_test, y_test_proba >= 0.50),
            f1_score(y_test, y_test_proba >= best_threshold),
        ],
    },
    index=["default_0.50", "train_f1_optimized"],
)

metrics

,threshold,roc_auc,accuracy,f1
default_0.50,0.500000,0.811481,0.733766,0.649573
train_f1_optimized,0.563438,0.811481,0.720779,0.605505


In [9]:
test_predictions = (y_test_proba >= best_threshold).astype(int)

print(f"Selected threshold: {best_threshold:.3f}")
print("\nConfusion matrix:")
print(confusion_matrix(y_test, test_predictions))
print("\nClassification report:")
print(classification_report(y_test, test_predictions, digits=3))

Selected threshold: 0.563

Confusion matrix:
[[78 22]
 [21 33]]

Classification report:
              precision    recall  f1-score   support

           0      0.788     0.780     0.784       100
           1      0.600     0.611     0.606        54

    accuracy                          0.721       154
   macro avg      0.694     0.696     0.695       154
weighted avg      0.722     0.721     0.721       154



## Model coefficients

The logistic regression coefficients are shown after the pipeline's cleaning and scaling steps. Positive coefficients increase the estimated diabetes probability; negative coefficients decrease it, all else equal.

In [10]:
final_lr = best_model.named_steps["model"]

coef_table = pd.DataFrame(
    {
        "feature": X.columns,
        "coefficient": final_lr.coef_.ravel(),
        "odds_ratio": np.exp(final_lr.coef_.ravel()),
        "abs_coefficient": np.abs(final_lr.coef_.ravel()),
    }
).sort_values("abs_coefficient", ascending=False)

coef_table

,feature,coefficient,odds_ratio,abs_coefficient
1,Glucose,1.135566,3.112936,1.135566
5,BMI,0.675677,1.965362,0.675677
0,Pregnancies,0.355601,1.427038,0.355601
6,DiabetesPedigreeFunction,0.275794,1.317576,0.275794
7,Age,0.184458,1.202567,0.184458
3,SkinThickness,0.023773,1.024058,0.023773
4,Insulin,-0.021422,0.978806,0.021422
2,BloodPressure,-0.000210,0.999790,0.000210


In [11]:
print("Final leakage-safe logistic regression pipeline:")
print(best_model)

Final leakage-safe logistic regression pipeline:
Pipeline(steps=[('zero_imputer',
                 ZeroToMedianImputer(columns=['Glucose', 'BloodPressure',
                                              'SkinThickness', 'Insulin',
                                              'BMI'])),
                ('scaler', StandardScaler()),
                ('model',
                 LogisticRegression(C=0.31622776601683794,
                                    class_weight='balanced', l1_ratio=0.1,
                                    max_iter=10000, penalty='elasticnet',
                                    random_state=42, solver='saga'))])
